# Logistic Regression in PySpark: The Simplest Version

**Goal:** predict whether a student will **Pass (1)** or **Fail (0)**.

**Features used (all numbers):** `StudyHours`, `Attendance`, `PreviousGrade`




## Steps
1. Import libraries
2. Start Spark
3. Load the CSV
4. Pick features and target
5. Combine features into one column
6. Split into train and test
7. Train the model
8. Make predictions
9. Check accuracy
10. Predict one new student

## Step 1: Import Libraries

- `SparkSession`: starts Spark
- `when`, `col`: used to turn "Pass"/"Fail" into 1/0
- `VectorAssembler`: puts all feature columns into one column (Spark needs this)
- `LogisticRegression`: the model
- `MulticlassClassificationEvaluator`: calculates accuracy

In [5]:
# Start-up tool for Spark
from pyspark.sql import SparkSession

# Helper functions to build columns
from pyspark.sql.functions import when, col

# Combines several columns into one "features" column
from pyspark.ml.feature import VectorAssembler

# The Logistic Regression model
from pyspark.ml.classification import LogisticRegression

# Tool to measure accuracy
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## Step 2: Start Spark

A Spark session is the entry point. Nothing works in PySpark without it.

In [6]:
# Create a Spark session (or reuse the existing one)
spark = SparkSession.builder.appName("SimpleLogisticRegression").getOrCreate()

print("Spark is ready, version:", spark.version)


Spark is ready, version: 4.0.5


## Step 3: Load the CSV

- `header=True`: the first row has column names
- `inferSchema=True`: Spark detects which columns are numbers

> **Colab users:** upload `students.csv` first, or use the full path such as `/content/students.csv`.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Read the CSV file into a Spark DataFrame
df = spark.read.csv("students1.csv", header=True, inferSchema=True)

# Look at the first 5 rows
df.show(5)

# Check the column types (numbers should show as int/double)
df.printSchema()

+----------+----------+-------------+------+
|StudyHours|Attendance|PreviousGrade|Result|
+----------+----------+-------------+------+
|       6.8|        41|           65|  Fail|
|       7.1|        97|           41|  Pass|
|       1.3|        45|           57|  Fail|
|       2.8|        81|           83|  Fail|
|       5.0|        57|           30|  Fail|
+----------+----------+-------------+------+
only showing top 5 rows
root
 |-- StudyHours: double (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- PreviousGrade: integer (nullable = true)
 |-- Result: string (nullable = true)



## Step 4: Pick Features and Target

- **Features (X):** the numeric columns the model learns from.
- **Target (y):** what we want to predict. The model needs a **number**, so we convert `Pass -> 1` and `Fail -> 0`.
- `Gender` is simply left out.

In [9]:
# Create the target column "label": Pass = 1, Fail = 0
df = df.withColumn("label", when(col("Result") == "Pass", 1).otherwise(0))

# Keep only the columns we need (Gender is dropped here)
df = df.select("StudyHours", "Attendance", "PreviousGrade", "label")

# Look at the result
df.show(5)

+----------+----------+-------------+-----+
|StudyHours|Attendance|PreviousGrade|label|
+----------+----------+-------------+-----+
|       6.8|        41|           65|    0|
|       7.1|        97|           41|    1|
|       1.3|        45|           57|    0|
|       2.8|        81|           83|    0|
|       5.0|        57|           30|    0|
+----------+----------+-------------+-----+
only showing top 5 rows


## Step 5: Combine Features into One Column

Spark's models want **one column that holds all features together** (a vector). `VectorAssembler` builds it.

`handleInvalid="skip"` tells it to ignore any row that has an empty value, so the model does not crash. (One row in this file has a missing Attendance.)

In [10]:
# Tell the assembler which columns are features
assembler = VectorAssembler(
    inputCols=["StudyHours", "Attendance", "PreviousGrade"],  # feature columns
    outputCol="features",                                      # new combined column
    handleInvalid="skip"                                       # skip rows with empty values
)

# Apply it to the data
df = assembler.transform(df)

# Show the original columns next to the combined column
df.select("StudyHours", "Attendance", "PreviousGrade", "features", "label").show(5, truncate=False)

+----------+----------+-------------+---------------+-----+
|StudyHours|Attendance|PreviousGrade|features       |label|
+----------+----------+-------------+---------------+-----+
|6.8       |41        |65           |[6.8,41.0,65.0]|0    |
|7.1       |97        |41           |[7.1,97.0,41.0]|1    |
|1.3       |45        |57           |[1.3,45.0,57.0]|0    |
|2.8       |81        |83           |[2.8,81.0,83.0]|0    |
|5.0       |57        |30           |[5.0,57.0,30.0]|0    |
+----------+----------+-------------+---------------+-----+
only showing top 5 rows


## Step 6: Split into Train and Test

- **Train (80%)**: the model learns from this.
- **Test (20%)**: hidden during training, used to check the model on **new** data.

`seed=42` makes the split the same every time you run it.

In [11]:
# Randomly split rows into training and testing parts
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# See how many rows are in each part
print("Training rows:", train_data.count())
print("Testing rows :", test_data.count())

Training rows: 33
Testing rows : 8


## Step 7: Train the Model

We create a `LogisticRegression` model and call `fit()` on the training data. `fit()` is where the learning happens.

- `featuresCol`: the combined feature column
- `labelCol`: the 0/1 answer column

In [12]:
# Create the model (not trained yet)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Train the model using the training data
model = lr.fit(train_data)

print("Model trained!")

Model trained!


## Step 8: Make Predictions

`transform()` adds new columns to the test data:

- `prediction`: the model's answer (0 = Fail, 1 = Pass)
- `probability`: `[chance of Fail, chance of Pass]`

In [13]:
# Use the trained model on the test data
predictions = model.transform(test_data)

# Compare the real answer (label) with the predicted answer (prediction)
predictions.select("StudyHours", "Attendance", "PreviousGrade",
                   "label", "prediction", "probability").show(truncate=False)

+----------+----------+-------------+-----+----------+------------------------------------------+
|StudyHours|Attendance|PreviousGrade|label|prediction|probability                               |
+----------+----------+-------------+-----+----------+------------------------------------------+
|1.7       |58        |76           |0    |0.0       |[0.9986655444295037,0.0013344555704962735]|
|2.5       |73        |84           |0    |0.0       |[0.8732372661526386,0.12676273384736136]  |
|2.9       |71        |80           |0    |0.0       |[0.894623888670241,0.10537611132975899]   |
|4.2       |66        |89           |1    |1.0       |[0.48179938037679654,0.5182006196232034]  |
|5.2       |56        |31           |0    |0.0       |[0.999130443102748,8.695568972519885E-4]  |
|5.6       |46        |68           |1    |0.0       |[0.9753633853758797,0.024636614624120257] |
|7.1       |74        |64           |1    |1.0       |[0.05672926837535546,0.9432707316246446]  |
|8.6       |89      

## Step 9: Check Accuracy

**Accuracy** = correct predictions / total predictions.

Example: 7 correct out of 8 = 0.875 (87.5%).

> The test set is very small, so treat the number as a learning example.

In [14]:
# Build an accuracy calculator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",           # the real answers
    predictionCol="prediction", # the model's answers
    metricName="accuracy"       # what to calculate
)

# Calculate accuracy on the test data
accuracy = evaluator.evaluate(predictions)

print("Accuracy:", round(accuracy, 3))

Accuracy: 0.875


## Step 10: Predict a New Student

Give the model a student it has never seen. We must build the `features` column in the same way as before.

**Example:** 7 study hours, 85% attendance, previous grade 75.

In [15]:
# Make a one-row DataFrame for the new student
new_student = spark.createDataFrame(
    [(7.0, 85, 75)],
    ["StudyHours", "Attendance", "PreviousGrade"]
)

# Build the same "features" column
new_student = assembler.transform(new_student)

# Predict
result = model.transform(new_student)

# prediction: 1 = Pass, 0 = Fail
result.select("prediction", "probability").show(truncate=False)

+----------+------------------------------------------+
|prediction|probability                               |
+----------+------------------------------------------+
|1.0       |[0.0029472419188206057,0.9970527580811794]|
+----------+------------------------------------------+



## Summary

| Step | Code | Purpose |
|---|---|---|
| Load | `spark.read.csv()` | Read the data |
| Target | `when(...)` | Pass/Fail -> 1/0 |
| Features | `VectorAssembler` | Combine columns into one |
| Split | `randomSplit()` | Train and test parts |
| Train | `lr.fit()` | Learn from data |
| Predict | `model.transform()` | Get predictions |
| Evaluate | `evaluator.evaluate()` | Get accuracy |



In [16]:
spark.stop()